# 🪟 NB-10: Sliding Window Attention – Longformer on Long Reasoning
**Goal:** Fine-tune Longformer with sliding window attention to process Claude's full thinking+response sequences.

**Modality:** Long-context Text | **Model:** allenai/longformer-base-4096

In [ ]:
!pip install transformers datasets -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
from transformers import LongformerTokenizer, LongformerForMaskedLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset
import torch

tok = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
model = LongformerForMaskedLM.from_pretrained("allenai/longformer-base-4096")

# Concatenate thinking + response as long documents
long_docs = []
for d in data:
    doc = f"QUESTION: {d['user']}\n\nREASONING: {d['thinking']}\n\nANSWER: {d['response']}"
    long_docs.append(doc)

def tokenize(batch):
    enc = tok(batch["text"], truncation=True, max_length=2048, padding="max_length")
    # Longformer needs global_attention_mask: 1 for [CLS]
    enc["global_attention_mask"] = [[1] + [0]*(len(ids)-1) for ids in enc["input_ids"]]
    return enc

collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=True, mlm_probability=0.15)
ds = Dataset.from_dict({"text": long_docs}).map(tokenize, batched=True, remove_columns=["text"]).train_test_split(0.1)

args = TrainingArguments("./longformer-claude", num_train_epochs=2,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    fp16=torch.cuda.is_available(), logging_steps=10, report_to="none")
Trainer(model=model, args=args, train_dataset=ds["train"],
        eval_dataset=ds["test"], data_collator=collator).train()
print("✅ Longformer sliding-window MLM complete!")
